# Name-Based Pairing

## What you'll learn

- When LINEAGE pairing breaks down and why NAME exists
- How NAME pairs artifacts by `original_name` stem across input roles
- How the same co-input edge machinery you saw with LINEAGE materializes pairings automatically
- The two failure modes you'll meet in production: unmatched stems (silent skip) and duplicate stems (raises)

**Prerequisites:** [Multi-Input Operations](04-multi-input-operations.ipynb)
**Estimated time:** 15 minutes

---

The previous tutorial paired multi-input streams by **lineage** — the framework
followed provenance ancestry to decide which artifacts go together. That works
when both streams share a common ancestor in the same pipeline.

What if they don't?


## When LINEAGE doesn't work

Imagine two independent ingest pipelines:

```
ingest A ──→ run.csv, sample_001.csv, sample_002.csv  (no common ancestor)
ingest B ──→ run.json, sample_001.json, sample_002.json
```

The two streams share no provenance ancestry — they were produced by separate
pipelines. But they share a **naming convention**: each `.csv` has a matching
`.json` for the same logical entity.

LINEAGE can't help here (no common ancestor exists). ZIP would only work by
accident (positional order). CROSS_PRODUCT would produce 9 pairings when only
3 are meaningful. NAME pairs by the shared filename stem.


## NAME pairing rules

`GroupByStrategy.NAME` pairs artifacts whose `original_name` stems match
exactly across roles. The rules:

- **Greedy extension stripping.** `data.tar.gz` and `data.csv` both reduce to
  the stem `data` — different formats of the same logical entity pair naturally.
- **Stems must be unique within each role.** Duplicates raise `ValueError` at
  the pairing phase.
- **A stem must appear in every role to produce a pairing.** Stems present in
  some roles but not all are skipped (logged at WARNING).
- **Case-sensitive.** `Sample_001` and `sample_001` do not match.


In [ ]:
from __future__ import annotations

import os
from enum import StrEnum
from pathlib import Path
from typing import Any, ClassVar

import polars as pl

from artisan.operations.base.operation_definition import OperationDefinition
from artisan.operations.curator import Merge
from artisan.operations.examples import DataGenerator
from artisan.orchestration import PipelineManager
from artisan.orchestration.runners import Runner
from artisan.schemas import ArtifactResult
from artisan.schemas.artifact.data import DataArtifact
from artisan.schemas.enums import GroupByStrategy
from artisan.schemas.execution.batch_strategy import BatchStrategy
from artisan.schemas.operation_config.runner_resources import RunnerResources
from artisan.schemas.specs.input_models import (
    ExecuteInput,
    PostprocessInput,
    PreprocessInput,
)
from artisan.schemas.specs.input_spec import InputSpec
from artisan.schemas.specs.output_spec import OutputSpec
from artisan.utils import tutorial_setup
from artisan.visualization import build_macro_graph, build_micro_graph

## Step 1: Two independent ingest streams

We'll simulate the independent-ingest scenario by running `DataGenerator`
twice with different seeds. Different seeds produce different file *contents*
(and therefore different content-addressed artifact IDs), but the filenames
follow the same `dataset_NNNNN.csv` convention — exactly the situation NAME is
designed for.


In [ ]:
env = tutorial_setup("name_basic")

pipeline = PipelineManager.create(
    name="name_basic",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# Two independent ingest streams. Same count → matching filenames. Different
# seeds → distinct content (and distinct artifact_ids).
pipeline.run(operation=DataGenerator, name="ingest_a", params={"count": 2, "seed": 42})
pipeline.run(operation=DataGenerator, name="ingest_b", params={"count": 2, "seed": 99})

Each ingest produced 2 artifacts. The filenames are deterministic
(`dataset_00000.csv`, `dataset_00001.csv`), so the two streams have matching
stems — but no shared ancestry, since they originated from separate steps.


## Step 2: Define a NAME-paired creator

The downstream operation takes both streams as inputs and pairs them by name.
The declaration is exactly as you saw in the LINEAGE tutorial — two input
roles, one output role — with one change: `group_by = GroupByStrategy.NAME`.

The `execute()` body deliberately reads bytes from **both** inputs into the
output. With CROSS_PRODUCT or NAME, paired execution units that produced
outputs depending on only one input would collide on content-addressed
artifact IDs.


In [ ]:
class DualInputName(OperationDefinition):
    """Pair two data streams by ``original_name`` stem."""

    name = "dual_input_name"
    description = "Name-paired join of two independent data streams"

    class InputRole(StrEnum):
        from_a = "from_a"
        from_b = "from_b"

    class OutputRole(StrEnum):
        result = "result"

    inputs: ClassVar[dict[str, InputSpec]] = {
        InputRole.from_a: InputSpec(artifact_type="data"),
        InputRole.from_b: InputSpec(artifact_type="data"),
    }
    outputs: ClassVar[dict[str, OutputSpec]] = {
        OutputRole.result: OutputSpec(
            artifact_type="data",
            infer_lineage_from={"inputs": ["from_a", "from_b"]},
        ),
    }
    group_by: GroupByStrategy | None = GroupByStrategy.NAME

    runner_resources: RunnerResources = RunnerResources(time_limit="00:10:00")
    batch_strategy: BatchStrategy = BatchStrategy(job_name="dual_input_name")

    def preprocess(self, inputs: PreprocessInput) -> dict[str, Any]:
        return {
            role: [a.materialized_path for a in artifacts]
            for role, artifacts in inputs.input_artifacts.items()
        }

    def execute(self, inputs: ExecuteInput) -> dict[str, Any]:
        out_dir = inputs.execute_dir
        os.makedirs(out_dir, exist_ok=True)

        from_a = inputs.inputs.get("from_a", [])
        from_b = inputs.inputs.get("from_b", [])
        if isinstance(from_a, (str, Path)):
            from_a = [from_a]
        if isinstance(from_b, (str, Path)):
            from_b = [from_b]

        a_path = Path(from_a[0])
        b_path = Path(from_b[0])
        # Output bytes depend on BOTH inputs so paired artifact_ids stay distinct.
        out_path = Path(out_dir) / f"{a_path.stem}.csv"
        out_path.write_bytes(a_path.read_bytes() + b"\n---\n" + b_path.read_bytes())
        return {}

    def postprocess(self, inputs: PostprocessInput) -> ArtifactResult:
        drafts = []
        for f in inputs.file_outputs:
            if f.endswith(".csv"):
                drafts.append(
                    DataArtifact.draft(
                        content=Path(f).read_bytes(),
                        original_name=os.path.basename(f),
                        step_number=inputs.step_number,
                    )
                )
        return ArtifactResult(success=True, artifacts={"result": drafts})

## Step 3: Run the paired step

Hand the two ingest outputs to `DualInputName` as separate roles. The framework
loads both streams' `original_name` values, intersects the stems, and
dispatches one execution unit per matched stem.


In [ ]:
pipeline.run(
    operation=DualInputName,
    name="join_by_name",
    inputs={
        "from_a": output("ingest_a", "datasets"),
        "from_b": output("ingest_b", "datasets"),
    },
    step_runner=Runner.LOCAL,
)
result = pipeline.finalize()
assert result["overall_success"]

## Inspecting the pairing

The macro graph shows the overall topology — two independent sources feeding
the NAME-paired step.


In [ ]:
build_macro_graph(env.delta_root)

The micro graph reveals the pairing edges. Each output of step 2 has
**two** incoming edges — one from `ingest_a`, one from `ingest_b`. NAME paired
them by filename stem, and the framework's existing co-input edge machinery
materialized the provenance edges automatically.


In [ ]:
build_micro_graph(env.delta_root)

### Verifying the pairing

Querying `artifact_edges` directly confirms the structure: each output has
two incoming edges that share a `group_id`, and the two output pairs have
distinct `group_id`s.


In [ ]:
edges = pl.read_delta(f"{env.delta_root}/provenance/artifact_edges")
join_edges = edges.filter(pl.col("target_artifact_type") == "data").filter(
    pl.col("source_role").is_in(["from_a", "from_b"])
)
join_edges.select(
    ["target_artifact_id", "source_artifact_id", "source_role", "group_id"]
).sort(["group_id", "source_role"])

## Failure modes

Two failures you'll meet eventually: stems present in one role but not the
other (silent skip with a WARNING) and duplicate stems within a single role
(raises `ValueError` at pairing time). Both are demonstrated below in
isolated pipelines so the main one above stays intact.


### Unmatched stems

If `ingest_b` has three artifacts but `ingest_a` only has two, the third stem
in `ingest_b` has no match in `ingest_a`. The framework logs a WARNING and
skips it — the step produces two outputs, not three.


In [ ]:
env_unmatched = tutorial_setup("name_unmatched")

pipeline_unmatched = PipelineManager.create(
    name="name_unmatched",
    delta_root=env_unmatched.delta_root,
    staging_root=env_unmatched.staging_root,
    working_root=env_unmatched.working_root,
)
out = pipeline_unmatched.output

pipeline_unmatched.run(
    operation=DataGenerator, name="ingest_a", params={"count": 2, "seed": 42}
)
pipeline_unmatched.run(
    operation=DataGenerator, name="ingest_b", params={"count": 3, "seed": 99}
)
pipeline_unmatched.run(
    operation=DualInputName,
    name="join_by_name",
    inputs={
        "from_a": out("ingest_a", "datasets"),
        "from_b": out("ingest_b", "datasets"),
    },
    step_runner=Runner.LOCAL,
)
result_unmatched = pipeline_unmatched.finalize()
assert result_unmatched["overall_success"]

# Two outputs, not three — dataset_00002 had no match in ingest_a and was skipped.
index = pl.read_delta(f"{env_unmatched.delta_root}/artifacts/index")
join_outputs = index.filter(pl.col("origin_step_number") == 2)
print(f"Outputs produced at step 2: {join_outputs.height}")

### Duplicate stems within a role

The uniqueness rule applies *within each role*. If you `Merge` two ingests
that produce overlapping filenames into a single role, that role now contains
duplicate stems. The pairing layer raises `ValueError` rather than make an
ambiguous match.


In [ ]:
env_dup = tutorial_setup("name_duplicates")

pipeline_dup = PipelineManager.create(
    name="name_duplicates",
    delta_root=env_dup.delta_root,
    staging_root=env_dup.staging_root,
    working_root=env_dup.working_root,
)
out = pipeline_dup.output

# Two ingests with the same count → identical filenames.
pipeline_dup.run(
    operation=DataGenerator, name="ingest_a1", params={"count": 2, "seed": 42}
)
pipeline_dup.run(
    operation=DataGenerator, name="ingest_a2", params={"count": 2, "seed": 100}
)
# Merge them into a single role → duplicate stems in that role.
pipeline_dup.run(
    operation=Merge,
    name="merged_a",
    inputs={
        "branch_a1": out("ingest_a1", "datasets"),
        "branch_a2": out("ingest_a2", "datasets"),
    },
)
# A second independent stream.
pipeline_dup.run(
    operation=DataGenerator, name="ingest_b", params={"count": 2, "seed": 7}
)

try:
    pipeline_dup.run(
        operation=DualInputName,
        name="join_by_name",
        inputs={
            "from_a": out("merged_a", "merged"),
            "from_b": out("ingest_b", "datasets"),
        },
        step_runner=Runner.LOCAL,
    )
    pipeline_dup.finalize()
except ValueError as exc:
    print(f"ValueError raised as expected:\n{exc}")

## Summary

`GroupByStrategy.NAME` pairs multi-input streams by `original_name` stem when
the streams have no shared ancestry. Use it when:

- Two (or more) ingest pipelines produce artifacts with matching filename
  conventions but no provenance link.
- LINEAGE doesn't apply, ZIP would be a positional accident, and
  CROSS_PRODUCT would explode.

Watch for:

- **Greedy extension stripping** — `run.log` and `run.cfg` both reduce to
  `run` and trigger the uniqueness check. Encode semantic suffixes in the
  base name (`run_log.txt`) when they need to coexist within one role.
- **Unmatched stems** are silently skipped; a count mismatch in your output
  is the visible signal.
- **Duplicate stems within a role** raise `ValueError` at pairing time.

## Next steps

- [Diamonds and Iteration](06-diamonds-and-iteration.ipynb) — diamond DAGs and iterative refinement loops
- [Operations Model](../../concepts/operations-model.md#pairing-strategies) — deeper conceptual treatment of pairing strategies
- [Writing Creator Operations](../../how-to-guides/writing-creator-operations.md) — using NAME (and other strategies) in your own ops
- [Step Overrides](../05-errors-and-control/01-step-overrides.ipynb) — overriding `group_by` per step
